# Fund Performance Analytics

## Objective
Analyze mutual fund performance by calculating daily returns and other financial metrics using the cleaned NAV history dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
nav = pd.read_csv("../data/processed/02_nav_history_cleaned.csv")

In [3]:
nav.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [4]:
nav.info()

<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  str    
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 1.1 MB


In [5]:
nav.describe()

,amfi_code,nav
count,46000.000000,46000.000000
mean,120247.000000,269.570265
std,14352.317221,577.187060
min,100016.000000,26.136600
25%,118632.750000,69.170425
50%,119551.500000,122.732150
75%,120842.250000,260.338675
max,149324.000000,4268.549700


## Data Inspection

The cleaned NAV history dataset was successfully loaded. Initial exploration was performed using `head()`, `info()`, and `describe()` to understand the dataset structure, data types, and summary statistics before calculating fund performance metrics.

In [6]:
nav["date"] = pd.to_datetime(nav["date"])

In [7]:
nav.info()

<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[us]
 2   nav        46000 non-null  float64       
dtypes: datetime64[us](1), float64(1), int64(1)
memory usage: 1.1 MB


In [8]:
nav = nav.sort_values(["amfi_code", "date"])

In [9]:
nav = nav.reset_index(drop=True)

In [10]:
nav.head(10)

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639
5,100016,2022-01-10,510.7136
6,100016,2022-01-11,513.5542
7,100016,2022-01-12,512.3195
8,100016,2022-01-13,510.2445
9,100016,2022-01-14,514.3636


In [11]:
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()

In [12]:
nav.head(10)

,amfi_code,date,nav,daily_return
0,100016,2022-01-03,520.4608,NaN
1,100016,2022-01-04,515.0971,-0.010306
2,100016,2022-01-05,521.7239,0.012865
3,100016,2022-01-06,515.7880,-0.011377
4,100016,2022-01-07,515.1639,-0.001210
5,100016,2022-01-10,510.7136,-0.008639
6,100016,2022-01-11,513.5542,0.005562
7,100016,2022-01-12,512.3195,-0.002404
8,100016,2022-01-13,510.2445,-0.004050
9,100016,2022-01-14,514.3636,0.008073


In [13]:
nav["daily_return"].describe()

count    45960.000000
mean         0.000631
std          0.010290
min         -0.058102
25%         -0.005042
50%          0.000340
75%          0.006324
max          0.064713
Name: daily_return, dtype: float64

In [14]:
first_nav = nav.groupby("amfi_code").first()
last_nav = nav.groupby("amfi_code").last()

In [15]:
first_nav.head()

,date,nav,daily_return
amfi_code,,,
100016,2022-01-03,520.4608,-0.010306
100025,2022-01-03,26.3169,-0.003553
100033,2022-01-03,107.3758,-0.013328
101206,2022-01-03,305.0996,0.001153
101207,2022-01-03,38.5736,-0.010865


In [16]:
last_nav.head()

,date,nav,daily_return
amfi_code,,,
100016,2026-05-29,583.6113,-0.012410
100025,2026-05-29,31.8843,-0.000809
100033,2026-05-29,342.0072,-0.004798
101206,2026-05-29,773.2939,-0.008529
101207,2026-05-29,53.9836,-0.014623


In [17]:
cagr = pd.DataFrame({
    "start_date": first_nav["date"],
    "end_date": last_nav["date"],
    "start_nav": first_nav["nav"],
    "end_nav": last_nav["nav"]
})

In [18]:
cagr["years"] = (cagr["end_date"] - cagr["start_date"]).dt.days / 365

In [19]:
cagr["CAGR"] = (
    (cagr["end_nav"] / cagr["start_nav"]) ** (1 / cagr["years"])
) - 1

In [20]:
cagr.head()

,start_date,end_date,start_nav,end_nav,years,CAGR
amfi_code,,,,,,
100016,2022-01-03,2026-05-29,520.4608,583.6113,4.40274,0.026352
100025,2022-01-03,2026-05-29,26.3169,31.8843,4.40274,0.044551
100033,2022-01-03,2026-05-29,107.3758,342.0072,4.40274,0.300997
101206,2022-01-03,2026-05-29,305.0996,773.2939,4.40274,0.235205
101207,2022-01-03,2026-05-29,38.5736,53.9836,4.40274,0.079331


In [21]:
cagr.describe()

,start_date,end_date,start_nav,end_nav,years,CAGR
count,40,40,40.000000,40.000000,40.00000,40.000000
mean,2022-01-03 00:00:00,2026-05-29 00:00:00,207.213793,357.613625,4.40274,0.167230
min,2022-01-03 00:00:00,2026-05-29 00:00:00,26.316900,31.884300,4.40274,0.011709
25%,2022-01-03 00:00:00,2026-05-29 00:00:00,51.871125,103.549550,4.40274,0.068570
50%,2022-01-03 00:00:00,2026-05-29 00:00:00,80.071800,188.070800,4.40274,0.165950
75%,2022-01-03 00:00:00,2026-05-29 00:00:00,175.525125,381.372750,4.40274,0.244696
max,2022-01-03 00:00:00,2026-05-29 00:00:00,3180.631800,4268.549700,4.40274,0.328016
std,NaN,NaN,499.020712,675.881823,0.00000,0.103008


In [22]:
cagr.sort_values("CAGR", ascending=False).head(10)

,start_date,end_date,start_nav,end_nav,years,CAGR
amfi_code,,,,,,
120505,2022-01-03,2026-05-29,135.8720,473.7640,4.40274,0.328016
119598,2022-01-03,2026-05-29,89.8738,309.2050,4.40274,0.323981
149324,2022-01-03,2026-05-29,81.6814,279.7511,4.40274,0.322621
148569,2022-01-03,2026-05-29,28.8620,97.7435,4.40274,0.319245
148567,2022-01-03,2026-05-29,70.2514,230.2708,4.40274,0.309499
120843,2022-01-03,2026-05-29,49.9131,163.2397,4.40274,0.308833
100033,2022-01-03,2026-05-29,107.3758,342.0072,4.40274,0.300997
149323,2022-01-03,2026-05-29,78.4622,245.3651,4.40274,0.295581
119094,2022-01-03,2026-05-29,68.3023,203.8581,4.40274,0.281926


In [23]:
cagr.to_csv("fund_cagr.csv")

In [24]:
mean_return = nav.groupby("amfi_code")["daily_return"].mean()

In [25]:
std_return = nav.groupby("amfi_code")["daily_return"].std()

In [26]:
sharpe = pd.DataFrame({
    "mean_return": mean_return,
    "std_return": std_return
})

In [27]:
risk_free_rate = 0.065 / 252

In [28]:
sharpe["Sharpe_Ratio"] = (
    (sharpe["mean_return"] - risk_free_rate)
    / sharpe["std_return"]
)

In [29]:
sharpe.head()

,mean_return,std_return,Sharpe_Ratio
amfi_code,,,
100016,0.000142,0.009164,-0.012694
100025,0.000170,0.002460,-0.035724
100033,0.001080,0.011929,0.068897
101206,0.000852,0.009177,0.064708
101207,0.000424,0.016251,0.010247


In [30]:
sharpe.sort_values("Sharpe_Ratio", ascending=False).head(10)

,mean_return,std_return,Sharpe_Ratio
amfi_code,,,
148567,0.001074,0.008941,0.091234
120843,0.001082,0.010008,0.082317
148569,0.001124,0.011134,0.077793
119551,0.000917,0.008656,0.076114
120505,0.001161,0.012152,0.074339
149323,0.001055,0.011179,0.071317
100033,0.001080,0.011929,0.068897
118632,0.000865,0.008913,0.068138
101206,0.000852,0.009177,0.064708


In [31]:
negative_returns = nav.copy()
negative_returns["downside_return"] = negative_returns["daily_return"].where(
    negative_returns["daily_return"] < 0
)

In [32]:
downside_std = negative_returns.groupby("amfi_code")["downside_return"].std()

In [33]:
sharpe["downside_std"] = downside_std

In [34]:
sharpe["Sortino_Ratio"] = (
    (sharpe["mean_return"] - risk_free_rate)
    / sharpe["downside_std"]
)

In [35]:
sharpe.sort_values("Sortino_Ratio", ascending=False).head(10)

,mean_return,std_return,Sharpe_Ratio,downside_std,Sortino_Ratio
amfi_code,,,,,
148567,0.001074,0.008941,0.091234,0.005428,0.150281
120843,0.001082,0.010008,0.082317,0.005531,0.148938
148569,0.001124,0.011134,0.077793,0.006404,0.135243
119551,0.000917,0.008656,0.076114,0.004887,0.134824
120505,0.001161,0.012152,0.074339,0.007067,0.127837
149323,0.001055,0.011179,0.071317,0.006750,0.118120
118632,0.000865,0.008913,0.068138,0.005211,0.116547
100033,0.001080,0.011929,0.068897,0.007133,0.115225
120504,0.000843,0.009048,0.064665,0.005145,0.113723


In [36]:
nav["running_max"] = nav.groupby("amfi_code")["nav"].cummax()

In [37]:
nav["drawdown"] = (nav["nav"] / nav["running_max"]) - 1

In [38]:
max_drawdown = nav.groupby("amfi_code")["drawdown"].min()

In [39]:
max_drawdown.head()

amfi_code
100016   -0.247344
100025   -0.043083
100033   -0.162172
101206   -0.112916
101207   -0.354469
Name: drawdown, dtype: float64

In [40]:
max_drawdown.sort_values().head(10)

amfi_code
119599   -0.525742
119095   -0.516778
101207   -0.354469
149324   -0.311719
119598   -0.287060
102886   -0.280011
100016   -0.247344
120842   -0.240035
118634   -0.233449
119093   -0.217514
Name: drawdown, dtype: float64

In [43]:
from scipy.stats import linregress

In [44]:
benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")
benchmark.head()

,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [45]:
benchmark["index_name"].unique()

<StringArray>
[        'NIFTY50',        'NIFTY100', 'NIFTY_MIDCAP150',    'BSE_SMALLCAP',
        'NIFTY500',   'CRISIL_LIQUID',     'CRISIL_GILT']
Length: 7, dtype: str

In [46]:
benchmark = benchmark[benchmark["index_name"] == "NIFTY100"].copy()
benchmark.head()

,date,index_name,close_value
1150,2022-01-03,NIFTY100,17778.24
1151,2022-01-04,NIFTY100,17537.52
1152,2022-01-05,NIFTY100,17607.73
1153,2022-01-06,NIFTY100,17556.05
1154,2022-01-07,NIFTY100,17664.02


In [47]:
benchmark["date"] = pd.to_datetime(benchmark["date"])

In [48]:
benchmark = benchmark.sort_values("date")

benchmark["benchmark_return"] = benchmark["close_value"].pct_change()

benchmark.head(10)

,date,index_name,close_value,benchmark_return
1150,2022-01-03,NIFTY100,17778.24,NaN
1151,2022-01-04,NIFTY100,17537.52,-0.013540
1152,2022-01-05,NIFTY100,17607.73,0.004003
1153,2022-01-06,NIFTY100,17556.05,-0.002935
1154,2022-01-07,NIFTY100,17664.02,0.006150
1155,2022-01-10,NIFTY100,17516.51,-0.008351
1156,2022-01-11,NIFTY100,17603.08,0.004942
1157,2022-01-12,NIFTY100,17763.76,0.009128
1158,2022-01-13,NIFTY100,17830.30,0.003746
1159,2022-01-14,NIFTY100,17578.93,-0.014098


In [49]:
benchmark = benchmark.sort_values("date")

benchmark["benchmark_return"] = benchmark["close_value"].pct_change()

benchmark.head(10)

,date,index_name,close_value,benchmark_return
1150,2022-01-03,NIFTY100,17778.24,NaN
1151,2022-01-04,NIFTY100,17537.52,-0.013540
1152,2022-01-05,NIFTY100,17607.73,0.004003
1153,2022-01-06,NIFTY100,17556.05,-0.002935
1154,2022-01-07,NIFTY100,17664.02,0.006150
1155,2022-01-10,NIFTY100,17516.51,-0.008351
1156,2022-01-11,NIFTY100,17603.08,0.004942
1157,2022-01-12,NIFTY100,17763.76,0.009128
1158,2022-01-13,NIFTY100,17830.30,0.003746
1159,2022-01-14,NIFTY100,17578.93,-0.014098


In [51]:
merged = nav.merge(
    benchmark[["date", "benchmark_return"]],
    on="date",
    how="inner"
)

merged.head()

,amfi_code,date,nav,daily_return,running_max,drawdown,benchmark_return
0,100016,2022-01-03,520.4608,NaN,520.4608,0.000000,NaN
1,100016,2022-01-04,515.0971,-0.010306,520.4608,-0.010306,-0.013540
2,100016,2022-01-05,521.7239,0.012865,521.7239,0.000000,0.004003
3,100016,2022-01-06,515.7880,-0.011377,521.7239,-0.011377,-0.002935
4,100016,2022-01-07,515.1639,-0.001210,521.7239,-0.012574,0.006150


In [52]:
merged = merged.dropna(subset=["daily_return", "benchmark_return"])

merged.head()

,amfi_code,date,nav,daily_return,running_max,drawdown,benchmark_return
1,100016,2022-01-04,515.0971,-0.010306,520.4608,-0.010306,-0.013540
2,100016,2022-01-05,521.7239,0.012865,521.7239,0.000000,0.004003
3,100016,2022-01-06,515.7880,-0.011377,521.7239,-0.011377,-0.002935
4,100016,2022-01-07,515.1639,-0.001210,521.7239,-0.012574,0.006150
5,100016,2022-01-10,510.7136,-0.008639,521.7239,-0.021104,-0.008351


In [53]:
merged = merged.dropna(subset=["daily_return", "benchmark_return"])

merged.head()

,amfi_code,date,nav,daily_return,running_max,drawdown,benchmark_return
1,100016,2022-01-04,515.0971,-0.010306,520.4608,-0.010306,-0.013540
2,100016,2022-01-05,521.7239,0.012865,521.7239,0.000000,0.004003
3,100016,2022-01-06,515.7880,-0.011377,521.7239,-0.011377,-0.002935
4,100016,2022-01-07,515.1639,-0.001210,521.7239,-0.012574,0.006150
5,100016,2022-01-10,510.7136,-0.008639,521.7239,-0.021104,-0.008351


In [54]:
alpha_beta = []

for fund, group in merged.groupby("amfi_code"):
    result = linregress(
        group["benchmark_return"],
        group["daily_return"]
    )

    alpha_beta.append({
        "amfi_code": fund,
        "Alpha": result.intercept * 252,
        "Beta": result.slope
    })

alpha_beta = pd.DataFrame(alpha_beta)

alpha_beta.head()

,amfi_code,Alpha,Beta
0,100016,0.037476,-0.058268
1,100025,0.042818,0.001158
2,100033,0.271954,0.005104
3,101206,0.213998,0.021086
4,101207,0.108971,-0.065289


In [56]:
scorecard = (
    cagr[["amfi_code", "CAGR"]]
    .merge(sharpe[["amfi_code", "Sharpe_Ratio"]], on="amfi_code")
    .merge(alpha_beta, on="amfi_code")
    .merge(max_drawdown.rename("Max_Drawdown"), on="amfi_code")
)

scorecard.head()

KeyError: "['amfi_code'] not in index"

In [57]:
cagr.head()

,start_date,end_date,start_nav,end_nav,years,CAGR
amfi_code,,,,,,
100016,2022-01-03,2026-05-29,520.4608,583.6113,4.40274,0.026352
100025,2022-01-03,2026-05-29,26.3169,31.8843,4.40274,0.044551
100033,2022-01-03,2026-05-29,107.3758,342.0072,4.40274,0.300997
101206,2022-01-03,2026-05-29,305.0996,773.2939,4.40274,0.235205
101207,2022-01-03,2026-05-29,38.5736,53.9836,4.40274,0.079331


In [58]:
cagr.columns

Index(['start_date', 'end_date', 'start_nav', 'end_nav', 'years', 'CAGR'], dtype='str')

In [59]:
cagr.index

Index([100016, 100025, 100033, 101206, 101207, 101208, 102885, 102886, 102887,
       118632, 118633, 118634, 118635, 118636, 119092, 119093, 119094, 119095,
       119120, 119551, 119552, 119598, 119599, 120503, 120504, 120505, 120506,
       120507, 120841, 120842, 120843, 120844, 125497, 125498, 148567, 148568,
       148569, 149322, 149323, 149324],
      dtype='int64', name='amfi_code')

In [60]:
cagr = cagr.reset_index()

In [61]:
cagr.head()

,amfi_code,start_date,end_date,start_nav,end_nav,years,CAGR
0,100016,2022-01-03,2026-05-29,520.4608,583.6113,4.40274,0.026352
1,100025,2022-01-03,2026-05-29,26.3169,31.8843,4.40274,0.044551
2,100033,2022-01-03,2026-05-29,107.3758,342.0072,4.40274,0.300997
3,101206,2022-01-03,2026-05-29,305.0996,773.2939,4.40274,0.235205
4,101207,2022-01-03,2026-05-29,38.5736,53.9836,4.40274,0.079331


In [62]:
sharpe.columns

Index(['mean_return', 'std_return', 'Sharpe_Ratio', 'downside_std',
       'Sortino_Ratio'],
      dtype='str')

In [63]:
alpha_beta.columns

Index(['amfi_code', 'Alpha', 'Beta'], dtype='str')

In [64]:
max_drawdown

amfi_code
100016   -0.247344
100025   -0.043083
100033   -0.162172
101206   -0.112916
101207   -0.354469
101208   -0.001622
102885   -0.108599
102886   -0.280011
102887   -0.215398
118632   -0.174141
118633   -0.186297
118634   -0.233449
118635   -0.116506
118636   -0.083164
119092   -0.144016
119093   -0.217514
119094   -0.209609
119095   -0.516778
119120   -0.043287
119551   -0.150124
119552   -0.118035
119598   -0.287060
119599   -0.525742
120503   -0.138979
120504   -0.125883
120505   -0.181885
120506   -0.198257
120507   -0.000977
120841   -0.175736
120842   -0.240035
120843   -0.129740
120844   -0.001163
125497   -0.152000
125498   -0.211173
148567   -0.112657
148568   -0.152726
148569   -0.163967
149322   -0.148446
149323   -0.172481
149324   -0.311719
Name: drawdown, dtype: float64

In [65]:
sharpe = sharpe.reset_index()

In [66]:
sharpe.head()

,amfi_code,mean_return,std_return,Sharpe_Ratio,downside_std,Sortino_Ratio
0,100016,0.000142,0.009164,-0.012694,0.005261,-0.022114
1,100025,0.000170,0.002460,-0.035724,0.001481,-0.059329
2,100033,0.001080,0.011929,0.068897,0.007133,0.115225
3,101206,0.000852,0.009177,0.064708,0.005238,0.113362
4,101207,0.000424,0.016251,0.010247,0.009555,0.017427


In [67]:
max_drawdown = max_drawdown.reset_index()
max_drawdown.columns = ["amfi_code", "Max_Drawdown"]

In [68]:
max_drawdown.head()

,amfi_code,Max_Drawdown
0,100016,-0.247344
1,100025,-0.043083
2,100033,-0.162172
3,101206,-0.112916
4,101207,-0.354469


In [69]:
scorecard = (
    cagr[["amfi_code", "CAGR"]]
    .merge(sharpe[["amfi_code", "Sharpe_Ratio"]], on="amfi_code")
    .merge(alpha_beta, on="amfi_code")
    .merge(max_drawdown, on="amfi_code")
)

scorecard.head()

,amfi_code,CAGR,Sharpe_Ratio,Alpha,Beta,Max_Drawdown
0,100016,0.026352,-0.012694,0.037476,-0.058268,-0.247344
1,100025,0.044551,-0.035724,0.042818,0.001158,-0.043083
2,100033,0.300997,0.068897,0.271954,0.005104,-0.162172
3,101206,0.235205,0.064708,0.213998,0.021086,-0.112916
4,101207,0.079331,0.010247,0.108971,-0.065289,-0.354469


In [70]:
scorecard.columns

Index(['amfi_code', 'CAGR', 'Sharpe_Ratio', 'Alpha', 'Beta', 'Max_Drawdown'], dtype='str')

In [72]:
scorecard.columns

Index(['amfi_code', 'CAGR', 'Sharpe_Ratio', 'Alpha', 'Beta', 'Max_Drawdown'], dtype='str')

In [73]:
import os

os.makedirs("../reports", exist_ok=True)

scorecard.to_csv("../reports/fund_scorecard.csv", index=False)

In [74]:
scorecard.head()

,amfi_code,CAGR,Sharpe_Ratio,Alpha,Beta,Max_Drawdown
0,100016,0.026352,-0.012694,0.037476,-0.058268,-0.247344
1,100025,0.044551,-0.035724,0.042818,0.001158,-0.043083
2,100033,0.300997,0.068897,0.271954,0.005104,-0.162172
3,101206,0.235205,0.064708,0.213998,0.021086,-0.112916
4,101207,0.079331,0.010247,0.108971,-0.065289,-0.354469


In [75]:
import os

os.makedirs("../reports", exist_ok=True)

scorecard.to_csv("../reports/fund_scorecard.csv", index=False)

print("Fund scorecard saved successfully!")

Fund scorecard saved successfully!
